# ConvNeXt Tiny Unfreeze Comparison (30 / 60 / 90)

This notebook keeps the dataset, preprocessing, augmentation, and optimizer setup fixed, and compares three stage-2 unfreezing depths: 30, 60, and 90 layers.

In [ ]:
import os
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from tensorflow import keras

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("TF version:", tf.__version__)
print("GPU:", gpus)


In [ ]:
def count_files_by_class(root_dir):
    counts = {}
    for class_name in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        counts[class_name] = sum(
            1
            for file_name in os.listdir(class_dir)
            if file_name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
        )
    return counts

def summarize_labels(ds, class_names):
    counts = Counter()
    for _, labels in ds:
        labels = labels.numpy().astype("int32").reshape(-1)
        counts.update(labels.tolist())
    return {class_names[idx]: counts.get(idx, 0) for idx in range(len(class_names))}

def collect_predictions(ds, model):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0).reshape(-1)
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(pb)
    return np.concatenate(y_true).astype("int32"), np.concatenate(y_prob)


In [ ]:
data_dir = "/mnt/e/DDSM/ROI"
assert os.path.exists(data_dir), f"Dataset path not found: {data_dir}"
print("Using data_dir:", data_dir)
print("Raw file counts by class:", count_files_by_class(data_dir))

img_size = (224, 224)
batch_size = 16
seed = 123
AUTOTUNE = tf.data.AUTOTUNE

os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
keras.utils.set_random_seed(seed)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

class_names = train_ds.class_names
print("Classes:", class_names)
print(f"Positive class for AUC/ROC: {class_names[1]}")

temp_card = tf.data.experimental.cardinality(temp_ds).numpy()
val_batches = max(1, temp_card // 2)
test_batches = temp_card - val_batches
if test_batches == 0:
    raise ValueError("Test split is empty. Reduce batch_size or use more data.")

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.prefetch(AUTOTUNE)

train_ds_prep = prepare(train_ds, shuffle=True)
val_ds_prep = prepare(val_ds)
test_ds_prep = prepare(test_ds)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())
print("Train label counts:", summarize_labels(train_ds, class_names))
print("Validation label counts:", summarize_labels(val_ds, class_names))
print("Test label counts:", summarize_labels(test_ds, class_names))


In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.10),
    keras.layers.RandomZoom(0.20),
    keras.layers.RandomTranslation(0.05, 0.05),
    keras.layers.RandomContrast(0.20),
], name="data_augmentation")

def build_convnext_model():
    backbone = keras.applications.ConvNeXtTiny(
        include_top=False,
        include_preprocessing=False,
        weights="imagenet",
        input_shape=img_size + (3,),
        pooling="avg",
        name="convnext_tiny",
    )
    preprocessing = keras.Sequential(
        [keras.layers.Rescaling(1.0 / 255)],
        name="preprocessing",
    )
    inputs = keras.Input(shape=img_size + (3,))
    x = preprocessing(inputs)
    x = data_augmentation(x)
    x = backbone(x, training=False)
    x = keras.layers.Dense(256, activation="gelu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="cancer_prob")(x)
    model = keras.Model(inputs, outputs)
    return model

model = build_convnext_model()
model.summary()


In [ ]:
backbone = model.get_layer("convnext_tiny")
backbone.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

stage1_trainable_weights = len(model.trainable_weights)
print("Stage 1 trainable weights:", stage1_trainable_weights)

stage1_ckpt = "/mnt/e/SW_training_outputs/checkpoints/best_convnext_stage1_comparison.keras"
cb1 = [
    keras.callbacks.ModelCheckpoint(
        stage1_ckpt,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

history1 = model.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=10,
    callbacks=cb1,
    verbose=1,
)


In [ ]:
unfreeze_candidates = [30, 60, 90]
comparison_results = []
comparison_histories = {}

for num_layers_to_unfreeze in unfreeze_candidates:
    print(f"\n===== Stage 2 experiment: unfreeze last {num_layers_to_unfreeze} layers =====")
    stage2_model = keras.models.load_model(stage1_ckpt, compile=False)
    stage2_backbone = stage2_model.get_layer("convnext_tiny")

    stage2_backbone.trainable = True
    for layer in stage2_backbone.layers:
        layer.trainable = False
    fine_tune_layers = stage2_backbone.layers[-num_layers_to_unfreeze:]
    for layer in fine_tune_layers:
        layer.trainable = True

    print("ConvNeXt backbone layers:", len(stage2_backbone.layers))
    print("Unfrozen last layers:", num_layers_to_unfreeze)
    print("Stage 2 trainable weights:", len(stage2_model.trainable_weights))

    stage2_model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    ckpt_path = f"/mnt/e/SW_training_outputs/checkpoints/best_convnext_unfreeze_{num_layers_to_unfreeze}.keras"
    cb2 = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    history2 = stage2_model.fit(
        train_ds_prep,
        validation_data=val_ds_prep,
        epochs=25,
        callbacks=cb2,
        verbose=1,
    )
    comparison_histories[num_layers_to_unfreeze] = history2

    best_model = keras.models.load_model(ckpt_path)
    best_model.compile(
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    test_metrics = best_model.evaluate(test_ds_prep, return_dict=True, verbose=1)
    test_true, test_prob = collect_predictions(test_ds_prep, best_model)
    test_pred = (test_prob >= 0.5).astype("int32")

    comparison_results.append({
        "unfreeze_layers": num_layers_to_unfreeze,
        "ckpt_path": ckpt_path,
        "best_val_auc": float(max(history2.history["val_auc"])),
        "test_auc": float(roc_auc_score(test_true, test_prob)),
        "test_acc": float(accuracy_score(test_true, test_pred)),
        "test_loss": float(test_metrics["loss"]),
        "test_precision": float(test_metrics["precision"]),
        "test_recall": float(test_metrics["recall"]),
    })

comparison_results = sorted(comparison_results, key=lambda item: item["test_auc"], reverse=True)
best_result = comparison_results[0]
best_unfreeze = best_result["unfreeze_layers"]
best_ckpt_path = best_result["ckpt_path"]
print("\nBest setting by test AUC:", best_result)


In [ ]:
print("ConvNeXt unfreeze comparison summary (sorted by test AUC):")
for result in comparison_results:
    print(
        f"unfreeze={result['unfreeze_layers']:>3} | "
        f"val_auc={result['best_val_auc']:.4f} | "
        f"test_auc={result['test_auc']:.4f} | "
        f"test_acc={result['test_acc']:.4f} | "
        f"precision={result['test_precision']:.4f} | "
        f"recall={result['test_recall']:.4f}"
    )

unfreeze_values = [item["unfreeze_layers"] for item in comparison_results]
test_auc_values = [item["test_auc"] for item in comparison_results]
test_acc_values = [item["test_acc"] for item in comparison_results]
val_auc_values = [item["best_val_auc"] for item in comparison_results]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(unfreeze_values, val_auc_values, marker="o", label="Best Val AUC")
plt.plot(unfreeze_values, test_auc_values, marker="o", label="Test AUC")
plt.xlabel("Unfrozen last layers")
plt.ylabel("AUC")
plt.title("AUC vs Unfreeze Depth")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(unfreeze_values, test_acc_values, marker="o", color="tab:green", label="Test Accuracy")
plt.xlabel("Unfrozen last layers")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Unfreeze Depth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_model = keras.models.load_model(best_ckpt_path)
best_model.compile(
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)
print("Loaded best comparison model from:", best_ckpt_path)
print("Best unfreeze depth:", best_unfreeze)

test_metrics = best_model.evaluate(test_ds_prep, return_dict=True)
print("Evaluation metrics:", test_metrics)

test_true, test_prob = collect_predictions(test_ds_prep, best_model)
test_pred = (test_prob >= 0.5).astype("int32")
test_fpr, test_tpr, _ = roc_curve(test_true, test_prob)
test_auc = roc_auc_score(test_true, test_prob)
test_acc = accuracy_score(test_true, test_pred)
cm = confusion_matrix(test_true, test_pred)

print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", classification_report(test_true, test_pred, target_names=class_names))

plt.figure(figsize=(6, 6))
plt.plot(test_fpr, test_tpr, label=f"ROC curve (AUC = {test_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (Best Depth = {best_unfreeze})")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

best_history = comparison_histories[best_unfreeze]
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(best_history.history["loss"], label="Train Loss")
plt.plot(best_history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(best_history.history["acc"], label="Train Accuracy")
plt.plot(best_history.history["val_acc"], label="Val Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(best_history.history["auc"], label="Train AUC")
plt.plot(best_history.history["val_auc"], label="Val AUC")
plt.title("AUC Curve")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
